# 01 · Descargar USU-Corn-WeedDB y crear mosaicos para descargar al computador

Este notebook descarga el dataset en `/content` y al final descarga al computador un `.zip` con los mosaicos generados.

Los mosaicos son artificiales: se crean pegando imágenes del dataset para probar la app con imágenes grandes, pero no son ortomosaicos reales.

In [ ]:
from pathlib import Path
import zipfile
import random
import requests
from PIL import Image
from tqdm.auto import tqdm

BASE_DIR = Path('/content/proyecto_malezas/datasets/usu')
DOWNLOAD_DIR = BASE_DIR / 'descargas'
EXTRACT_DIR = BASE_DIR / 'USU-Corn-WeedDB'
OUTPUT_DIR = BASE_DIR / 'mosaicos_usu'

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('BASE_DIR:', BASE_DIR)

In [ ]:
ZENODO_RECORD_ID = '20044178'
API_URL = f'https://zenodo.org/api/records/{ZENODO_RECORD_ID}'

response = requests.get(API_URL, timeout=60)
if response.status_code != 200:
    raise RuntimeError(f'No se pudo acceder a Zenodo. Código: {response.status_code}')

record = response.json()
files = record.get('files', [])

print('Archivos encontrados:')
for i, f in enumerate(files, start=1):
    size_mb = f.get('size', 0) / (1024 * 1024)
    print(f'{i}. {f.get("key")} - {size_mb:.2f} MB')

In [ ]:
def descargar_archivo(url, destino):
    destino = Path(destino)
    if destino.exists() and destino.stat().st_size > 0:
        print(f'Ya existe: {destino.name}')
        return destino

    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        with open(destino, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True, desc=destino.name) as pbar:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
    return destino

archivos_descargados = []
for f in files:
    nombre = f.get('key')
    enlace = f.get('links', {}).get('self')
    if enlace is None:
        continue
    archivos_descargados.append(descargar_archivo(enlace, DOWNLOAD_DIR / nombre))

print('Descarga terminada.')

In [ ]:
for archivo in archivos_descargados:
    if archivo.suffix.lower() == '.zip':
        print('Descomprimiendo:', archivo.name)
        with zipfile.ZipFile(archivo, 'r') as z:
            z.extractall(EXTRACT_DIR)

print('Contenido principal:')
for p in sorted(EXTRACT_DIR.iterdir()):
    print('-', p.name)

In [ ]:
extensiones = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']
imagenes = []
for ext in extensiones:
    imagenes.extend(EXTRACT_DIR.rglob(ext))
imagenes = sorted(imagenes)
print('Total de imágenes encontradas:', len(imagenes))
for img in imagenes[:10]:
    print(img)

In [ ]:
def filtrar_por_fragmento(rutas, fragmentos):
    salida = []
    for ruta in rutas:
        ruta_str = str(ruta).replace('\\', '/').lower()
        if any(fragmento.lower() in ruta_str for fragmento in fragmentos):
            salida.append(ruta)
    return salida

grupos = {
    'test': filtrar_por_fragmento(imagenes, ['images/test', 'test/images', '/test/']),
    'val': filtrar_por_fragmento(imagenes, ['images/val', 'val/images', '/val/']),
    'train': filtrar_por_fragmento(imagenes, ['images/train', 'train/images', '/train/']),
    'todas': imagenes
}

for nombre, rutas in grupos.items():
    print(nombre, len(rutas))

In [ ]:
def crear_mosaico(rutas_imagenes, salida, filas=5, columnas=5, tam=640, semilla=42):
    rutas_imagenes = list(rutas_imagenes)
    if len(rutas_imagenes) == 0:
        raise ValueError('No hay imágenes para crear el mosaico.')

    random.seed(semilla)
    total = filas * columnas
    seleccionadas = random.sample(rutas_imagenes, total) if len(rutas_imagenes) >= total else [random.choice(rutas_imagenes) for _ in range(total)]
    lienzo = Image.new('RGB', (columnas * tam, filas * tam), color=(255, 255, 255))

    for idx, ruta in enumerate(seleccionadas):
        img = Image.open(ruta).convert('RGB').resize((tam, tam))
        fila = idx // columnas
        columna = idx % columnas
        lienzo.paste(img, (columna * tam, fila * tam))

    salida = Path(salida)
    salida.parent.mkdir(parents=True, exist_ok=True)
    lienzo.save(salida, quality=95)
    return salida

salidas = []
for nombre_grupo in ['test', 'val', 'train', 'todas']:
    rutas = grupos[nombre_grupo]
    if len(rutas) == 0:
        continue
    salidas.append(crear_mosaico(rutas, OUTPUT_DIR / f'mosaico_{nombre_grupo}_5x5.png', filas=5, columnas=5, tam=640, semilla=42))
    salidas.append(crear_mosaico(rutas, OUTPUT_DIR / f'mosaico_{nombre_grupo}_10x10.png', filas=10, columnas=10, tam=640, semilla=43))

print('Mosaicos creados:')
for s in salidas:
    print(s)

In [ ]:
zip_salida = BASE_DIR / 'mosaicos_usu_generados.zip'
if zip_salida.exists():
    zip_salida.unlink()

with zipfile.ZipFile(zip_salida, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for archivo in OUTPUT_DIR.rglob('*'):
        if archivo.is_file():
            z.write(archivo, archivo.relative_to(OUTPUT_DIR.parent))

print('ZIP generado:', zip_salida)
print('Tamaño MB:', zip_salida.stat().st_size / (1024 * 1024))

In [ ]:
from google.colab import files
files.download(str(zip_salida))